# DDR Chart Generator — Training Notebook

**Make sure to set Runtime → Change runtime type → T4 GPU before running!**

This notebook:
1. Installs dependencies
2. Downloads the DDC dataset
3. Trains the CNN+Transformer model with curriculum learning
4. Generates a chart from a test song
5. Produces a loss curve visualization

In [ ]:
# ── 1. Clone the project repo ──────────────────────────────────────────
# Replace with your actual GitHub repo URL after you push the code
!git clone https://github.com/YOUR_USERNAME/ddc-chart-gen.git
%cd ddc-chart-gen

In [ ]:
# ── 2. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torch torchvision torchaudio --quiet

In [ ]:
# ── 3. Download DDC dataset ─────────────────────────────────────────────
# The DDC repo provides a download script
!git clone https://github.com/chrisdonahue/ddc.git ddc_repo

# Follow the DDC repo instructions to download the dataset
# The data is at: https://github.com/chrisdonahue/ddc#data
# Download and unpack to data/ddc/
!mkdir -p data/ddc

# If you have a direct download link to the .tar.gz:
# !wget -O data/ddc.tar.gz "YOUR_DOWNLOAD_LINK"
# !tar -xzf data/ddc.tar.gz -C data/ddc --strip-components=1

print('Dataset directory structure:')
!ls data/ddc | head -20

In [ ]:
# ── 4. Verify dataset ───────────────────────────────────────────────────
import os
from pathlib import Path

data_root = 'data/ddc'
song_dirs = [d for d in Path(data_root).iterdir() if d.is_dir()]
print(f'Found {len(song_dirs)} song directories')

# Check one song
if song_dirs:
    sample = song_dirs[0]
    print(f'\nSample: {sample.name}')
    for f in sample.iterdir():
        print(f'  {f.name}')

In [ ]:
# ── 5. Test data processing on one song ────────────────────────────────
import sys
sys.path.insert(0, '.')

from utils.data_utils import build_sample, parse_sm_file

# Find a sample song
test_dir = song_dirs[0]
sm_files = list(test_dir.glob('*.sm'))
audio_files = [f for f in test_dir.iterdir() if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]

if sm_files and audio_files:
    sm_data = parse_sm_file(str(sm_files[0]))
    print(f"Title: {sm_data['title']}")
    print(f"BPMs: {sm_data['bpms']}")
    print(f"Charts: {[(c['difficulty'], c['meter']) for c in sm_data['charts']]}")
    
    sample = build_sample(str(audio_files[0]), str(sm_files[0]))
    if sample:
        print(f"\nX shape: {sample['X'].shape}")
        print(f"y shape: {sample['y'].shape}")
        print(f"Difficulty: {sample['difficulty']}")
        print(f"Step density: {(sample['y'].sum(-1) > 0).mean():.3f}")

In [ ]:
# ── 6. Train ────────────────────────────────────────────────────────────
# Adjust epochs_per_stage based on time budget:
#   Quick test:     epochs_per_stage=5
#   Full training:  epochs_per_stage=30

!python train.py \
    --data_root data/ddc \
    --epochs_per_stage 20 \
    --batch_size 32 \
    --d_model 256 \
    --n_layers 4 \
    --curriculum_start 0 \
    --checkpoint_dir checkpoints

In [ ]:
# ── 7. Plot loss curves ─────────────────────────────────────────────────
import json
import matplotlib.pyplot as plt
import numpy as np

with open('logs/train_history.json') as f:
    history = json.load(f)

train_h = history['train']
val_h   = history['val']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Color by curriculum stage
stage_colors = {0:'#3498db', 1:'#2ecc71', 2:'#f39c12', 3:'#e74c3c', 4:'#9b59b6'}

for metric, ax, title in [
    ('loss',       axes[0], 'Total Loss'),
    ('step_loss',  axes[1], 'Step Placement Loss'),
    ('f1',         axes[2], 'Step F1 Score'),
]:
    for stage in range(5):
        t_vals = [x[metric] for x in train_h if x['stage'] == stage]
        v_vals = [x[metric] for x in val_h   if x['stage'] == stage]
        if not t_vals:
            continue
        color = stage_colors[stage]
        xs = range(len(t_vals))
        ax.plot(xs, t_vals, color=color, alpha=0.8, label=f'Stage {stage} train')
        ax.plot(xs, v_vals, color=color, alpha=0.5, linestyle='--', label=f'Stage {stage} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: logs/training_curves.png')

In [ ]:
# ── 8. Generate a chart from a test song ───────────────────────────────
# Upload your own .mp3 or use one from the dataset

# Option A: use a song from the dataset
test_audio = str(audio_files[0])  # from the sample song above

# Option B: upload your own file
# from google.colab import files
# uploaded = files.upload()
# test_audio = list(uploaded.keys())[0]

!python generate.py \
    --audio {test_audio} \
    --checkpoint checkpoints/best_model.pt \
    --difficulty 2 \
    --threshold 0.5 \
    --output output_chart

In [ ]:
# ── 9. Download results ─────────────────────────────────────────────────
from google.colab import files

# Download the visualizer
files.download('output_chart/visualizer.html')

# Download the .sm chart
files.download('output_chart/chart.sm')

# Download the model checkpoint
files.download('checkpoints/best_model.pt')

# Download training curves
files.download('logs/training_curves.png')

In [ ]:
# ── 10. Threshold sweep (difficulty knob exploration) ──────────────────
# Explore how the step threshold affects chart density

import torch
import numpy as np
import matplotlib.pyplot as plt

from models.model import DDRTransformer, generate_chart
from generate import audio_to_model_input

ckpt = torch.load('checkpoints/best_model.pt', map_location='cpu')
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
X = audio_to_model_input(test_audio)

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []

for th in thresholds:
    mask, _ = generate_chart(model, X, difficulty=2, step_threshold=th)
    densities.append(mask.mean())

plt.figure(figsize=(7, 4))
plt.plot(thresholds, densities, 'o-', color='#e74c3c')
plt.xlabel('Step threshold')
plt.ylabel('Step density (fraction of active timesteps)')
plt.title('Difficulty knob: threshold vs. chart density')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logs/threshold_sweep.png', dpi=150)
plt.show()